In [ ]:
from pathlib import Path
import json, random, sys, time, os

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.wajib.rnn.RNN import RNNScratch
from src.wajib.lstm.LSTM import LSTMScratch, ImageCaptioningPipeline
from src.wajib.shared.layers import EmbeddingLayer, DenseLayer
from src.wajib.shared.preprocessing import (
    loadFlickr8kCaptions, loadVocabulary, cleanCaption, buildVocabulary, saveVocabulary, loadVocabulary, tokenizeCaption, padSequences
)
from src.wajib.shared.decoder import greedyDecode

# Untuk bleu dan meteor
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

I0000 00:00:1778842406.629494   35978 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778842406.631726   35978 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778842406.947118   35978 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778842413.635802   35978 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

In [2]:
FEATURES_NPY  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_features.npy"
FEATURES_IDX  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_index.json"
CAPTIONS_FILE = PROJECT_ROOT / "data/flickr8k/captions.txt"
VOCAB_PATH    = PROJECT_ROOT / "src/wajib/weights/vocab.json"
DATA_DIR      = PROJECT_ROOT / "data/flickr8k"
FLICKR_DIR    = DATA_DIR / "Images"

RNN_WEIGHTS_DIR   = PROJECT_ROOT / "src/wajib/weights/rnn"
LSTM_WEIGHTS_DIR  = PROJECT_ROOT / "src/wajib/weights/lstm"
CNN_ENCODER_PATH  = PROJECT_ROOT / "src/wajib/weights/cnn"

EMBED_DIM    = 256
MAX_LEN      = 30
CNN_FEAT_DIM = 2048
TARGET_SIZE  = (299, 299)

features_matrix = np.load(FEATURES_NPY)
with open(FEATURES_IDX) as f:
    idxNames = json.load(f)

imageFeatures = {name: features_matrix[i] for i, name in enumerate(idxNames)}
print(f"Fitur: {len(imageFeatures)} gambar")

captionsDict = loadFlickr8kCaptions(str(CAPTIONS_FILE))
vocab = loadVocabulary(str(VOCAB_PATH))
id2word = {v: k for k, v in vocab.items()}
print(f"Caption: {len(captionsDict)} gambar")
print(f"Kosa kata: {len(vocab)} token")

# Bagi ulang data (sama seperti training)
allImages = list(captionsDict.keys())
random.seed(SEED)
random.shuffle(allImages)

trainImgs = set(allImages[:6000])
valImgs   = set(allImages[6000:7000])
testImgs  = set(allImages[7000:])

testCaps = {k: v for k, v in captionsDict.items() if k in testImgs}
print(f"Data uji: {len(testCaps)} gambar")

Fitur: 8091 gambar
Caption: 8091 gambar
Kosa kata: 4558 token
Data uji: 1091 gambar


In [ ]:
class RNNPipeline:
    def __init__(self, rnn, proj, embed, out, vocab, id2word, max_len=30):
        self.rnn = rnn
        self.proj = proj
        self.embed = embed
        self.out = out
        self.vocab = vocab
        self.id2word = id2word
        self.max_len = max_len

    def generateCaption(self, feat):
        hyp = greedyDecode(
            self.rnn, self.proj, self.embed, self.out, 
            feat, self.vocab, self.max_len
        )
        return ' '.join(hyp)

def buildRNNPipeline(modelPath, numLayers, vocab_path, max_len=30):
    vocab = loadVocabulary(vocab_path)
    id2word = {v: k for k, v in vocab.items()}

    model = tf.keras.models.load_model(str(modelPath))

    embed = EmbeddingLayer()
    embed.loadWeights(model.get_layer('embedding'))

    proj = DenseLayer()
    proj.loadWeights(model.get_layer('cnn_proj'))

    out = DenseLayer(activation='softmax')
    out.loadWeights(model.get_layer('output'))

    # RNN OR LSTM
    if 'lstm' in modelPath.name.lower():
        rnn = LSTMScratch()
        prefix = 'lstm'
    else:
        rnn = RNNScratch()
        prefix = 'rnn'

    rnn.loadWeights([
        model.get_layer(f'{prefix}_{i}')
        for i in range(numLayers)
    ])

    return RNNPipeline(rnn, proj, embed, out, vocab, id2word, max_len)

### Evaluation Loop

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

smooth = SmoothingFunction().method1


def evaluatePipeline(name, pipeline, testCaps):

    hypotheses = []
    references = []

    t0 = time.time()

    for i, (imgName, caps) in enumerate(testCaps.items()):

        if i % 200 == 0:
            print(
                f"{name}: {i}/{len(testCaps)}",
                end='\r'
            )

        if imgName not in imageFeatures:
            continue

        feat = imageFeatures[imgName]

        hyp = pipeline.generateCaption(feat)

        hypotheses.append(hyp.split())

        references.append([
            cap.split()
            for cap in caps
        ])

    elapsed = time.time() - t0

    bleu4 = corpus_bleu(
        references,
        hypotheses,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    meteorScores = []

    for refs, hyp in zip(references, hypotheses):

        meteorScores.append(
            max([
                meteor_score([ref], hyp)
                for ref in refs
            ])
        )

    meteor = np.mean(meteorScores)

    print(
        f"{name:<22} "
        f"BLEU-4={bleu4:.4f} "
        f"METEOR={meteor:.4f} "
        f"time={elapsed:.1f}s"
    )

    return {
        'bleu4': bleu4,
        'meteor': meteor,
        'time_s': elapsed,
    }

## Results Final

In [ ]:
allResults = {}

allModels = [
    ('RNN', rnnModels),
    ('LSTM', lstmModels),
]

for modelType, modelList in allModels:

    print("\n" + "=" * 70)
    print(f"{modelType} EXPERIMENTS")
    print("=" * 70)

    for modelPath in modelList:

        name = modelPath.stem

        numLayers = int(
            name.split('L')[0].split('_')[1]
        )

        try:

            pipeline = buildRNNPipeline(
                modelPath=modelPath,
                numLayers=numLayers,
                vocab_path=VOCAB_PATH,
                max_len=MAX_LEN
            )

            result = evaluatePipeline(
                name,
                pipeline,
                testCaps
            )

            result['type'] = modelType

            allResults[name] = result

        except Exception as e:

            print(f"Gagal di {name}: {e}")

In [ ]:
# Visualisasi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Perbandingan BLEU-4
ax = axes[0, 0]
models = sorted(allResults.keys())
bleus = [allResults[m]['bleu4'] for m in models]
types = ['RNN' if allResults[m]['type'] == 'RNN' else 'LSTM' for m in models]
colors = ['#1f77b4' if t == 'RNN' else '#ff7f0e' for t in types]

ax.barh(range(len(models)), bleus, color=colors)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Skor BLEU-4')
ax.set_title('Skor BLEU-4: 12 Variasi')
ax.grid(axis='x', alpha=0.3)

# Plot 2: Perbandingan METEOR
ax = axes[0, 1]
meteors = [allResults[m]['meteor'] for m in models]
ax.barh(range(len(models)), meteors, color=colors)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Skor METEOR')
ax.set_title('Skor METEOR: 12 Variasi')
ax.grid(axis='x', alpha=0.3)

# Plot 3: Distribusi BLEU-4 per arsitektur
ax = axes[1, 0]
x = ['RNN', 'LSTM']
rnn_bleus = [allResults[n]['bleu4'] for n in rnnResults]
lstm_bleus = [allResults[n]['bleu4'] for n in lstmResults]

positions = [0, 1]
bp = ax.boxplot([rnn_bleus, lstm_bleus], positions=positions, widths=0.6, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.scatter([0]*len(rnn_bleus), rnn_bleus, alpha=0.5, s=50, color='#1f77b4')
ax.scatter([1]*len(lstm_bleus), lstm_bleus, alpha=0.5, s=50, color='#ff7f0e')

ax.set_xticklabels(x)
ax.set_ylabel('BLEU-4')
ax.set_title('Distribusi BLEU-4: RNN vs LSTM')
ax.grid(axis='y', alpha=0.3)

# Plot 4: Waktu inferensi
ax = axes[1, 1]
times = [allResults[m]['time_s'] for m in models]
ax.barh(range(len(models)), times, color=colors)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Waktu inferensi (detik)')
ax.set_title('Waktu Inferensi pada Data Uji')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "src/wajib/weights/pipeline_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("Visualisasi tersimpan")

### Ringkasan Visualisasi 4 Panel

**Panel 1 (kiri atas): Skor BLEU-4** — bar horizontal untuk 12 model, biru=RNN, oranye=LSTM.
**Panel 2 (kanan atas): Skor METEOR** — metrik semantik yang melengkapi BLEU.
**Panel 3 (kiri bawah): Box plot RNN vs LSTM** — sebaran BLEU-4 tiap arsitektur.
**Panel 4 (kanan bawah): Waktu inferensi** — membandingkan biaya komputasi.

**Inti:** LSTM biasanya unggul pada kualitas, namun butuh waktu inferensi lebih lama.

In [ ]:
print("\n" + "="*70)
print("VARIASI PANJANG MAKSIMUM CAPTION (MODEL LSTM TERBAIK)")
print("="*70)

maxLenResults = {}

bestLstmPath = LSTM_WEIGHTS_DIR / f"{bestLstm}.keras"
model = tf.keras.models.load_model(str(bestLstmPath))

embed = EmbeddingLayer()
embed.loadWeights(model.get_layer('embedding'))

proj = DenseLayer()
proj.loadWeights(model.get_layer('cnn_proj'))

out = DenseLayer(activation='softmax')
out.loadWeights(model.get_layer('output'))

numLayersBest = int(bestLstm.split('L')[0].split('_')[1])
lstm = LSTMScratch()
lstm.loadWeights([model.get_layer(f'lstm_{i}') for i in range(numLayersBest)])

# Uji beberapa nilai max_len
maxLenVariations = [10, 20, 30, 40, 50]

for maxLenTest in maxLenVariations:
    hypotheses = []
    references = []
    
    for imgName, caps in testCaps.items():
        if imgName not in imageFeatures:
            continue
        
        feat = imageFeatures[imgName]
        
        # Buat caption dengan max_len yang berbeda
        x = proj.forward(feat)
        h = [np.zeros(cell.hiddenDim) for cell in lstm.cells]
        c = [np.zeros(cell.hiddenDim) for cell in lstm.cells]
        
        words = []
        token = vocab['<start>']
        
        for _ in range(maxLenTest):
            x_emb = embed.forward(token)
            h, c = lstm.forwardStep(x_emb, h, c)
            logits = out.forward(h[-1])
            token = int(np.argmax(logits))
            
            if token == vocab['<end>']:
                break
            words.append(id2word.get(token, '<unk>'))
        
        hyp = ' '.join(words)
        hypotheses.append(hyp.split())
        references.append([cap.split() for cap in caps])
    
    # Hitung BLEU-4
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))
    maxLenResults[maxLenTest] = bleu4
    print(f"  max_len={maxLenTest:2d}  ->  BLEU-4={bleu4:.4f}")

# Plot max_len vs BLEU-4
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
maxLens = sorted(maxLenResults.keys())
bleus = [maxLenResults[ml] for ml in maxLens]

ax.plot(maxLens, bleus, marker='o', linewidth=2, markersize=8, color='#ff7f0e')
ax.fill_between(maxLens, bleus, alpha=0.3, color='#ff7f0e')
ax.set_xlabel('Panjang Maksimum Caption')
ax.set_ylabel('Skor BLEU-4')
ax.set_title(f'Pengaruh Panjang Maksimum Caption terhadap BLEU-4\n(Model LSTM terbaik: {bestLstm})')
ax.grid(True, alpha=0.3)
ax.set_xticks(maxLens)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "src/wajib/weights/maxlen_variation.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPanjang maksimum terbaik: {max(maxLenResults, key=maxLenResults.get)} (BLEU-4={max(maxLenResults.values()):.4f})")


### Eksplorasi: Pengaruh Panjang Maksimum Caption

Menggunakan model LSTM terbaik, kita uji beberapa nilai max_len: [10, 20, 30, 40, 50].
Tujuannya melihat trade-off antara caption yang terlalu pendek dan terlalu panjang.

**Pola yang umum terjadi:**
- max_len kecil -> caption terpotong, skor BLEU rendah
- max_len sedang (30-40) -> hasil paling stabil
- max_len besar -> caption terlalu panjang, skor bisa turun

**Catatan:**
- Rata-rata caption Flickr8k sekitar 11-12 kata
- max_len=30 sudah cukup aman untuk training
- Uji ini membantu memastikan model tidak terlalu sensitif terhadap batas panjang

## Kesimpulan Notebook 5: Evaluasi Pipeline Penuh

**Temuan utama:**
1. **LSTM lebih baik dari RNN** dalam kualitas caption
2. **Kedalaman optimal**: 2-3 layer dengan hidden 256-512 biasanya paling seimbang
3. **Model terbaik** sering berada di variasi LSTM dengan hidden besar
4. **Biaya inferensi** LSTM lebih tinggi, tetapi masih layak untuk evaluasi offline
5. **Panjang caption** optimal berada di kisaran 30-40

**Interpretasi metrik:**
- BLEU-4 sensitif terhadap kecocokan kata
- METEOR lebih peka pada makna
- Keduanya saling melengkapi

**Validasi:**
- LSTM scratch mampu meniru model Keras
- Greedy decoding sudah cukup untuk evaluasi awal
- CNN encoder memberi sinyal awal yang kuat

**Langkah berikutnya (Notebook 6):** Analisis kualitatif pada contoh gambar dari tiap kuartil BLEU.